# 🤖 PM Lifecycle Agent
### End-to-end product management workflow powered by Claude API

**Author:** Shruthi Khurana  
**Stack:** Python · Anthropic Claude API · Human-in-the-Loop  

This agent walks through the full PM product lifecycle — from raw problem statement to GTM plan — with human checkpoint interventions at every stage. Each stage builds context from the previous one, so the outputs compound in quality as you progress.

---
**Stages:**
1. 🔍 Discovery & Ideation
2. 🗺️ Roadmapping & Prioritization
3. 📋 BRD / FRD Generation
4. 🎨 UI/UX & Design Brief
5. 🧪 UAT Planning & Test Cases
6. 🚀 Production Readiness
7. 📣 GTM Planning

---
> **How to use:** Run cells top to bottom. At each `[HUMAN CHECKPOINT]` cell, review the output, edit if needed, then confirm to proceed.

## ⚙️ Setup

In [ ]:
import anthropic
import os
import json
import textwrap
from datetime import datetime
from IPython.display import display, Markdown

# ── Configuration ──────────────────────────────────────────────
MODEL = "claude-opus-4-6"          # Use latest Claude model
MAX_TOKENS = 4096
OUTPUT_DIR = "./artifacts"         # All outputs saved here

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Client ─────────────────────────────────────────────────────
# Set your API key: export ANTHROPIC_API_KEY='your-key'
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# ── Shared context store (builds across stages) ────────────────
context = {}

# ── Helper functions ───────────────────────────────────────────
def call_claude(system_prompt: str, user_prompt: str) -> str:
    """Single Claude API call. Returns text response."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )
    return message.content[0].text

def display_output(stage: str, content: str):
    """Pretty-print stage output as markdown."""
    display(Markdown(f"### {stage}\n\n{content}"))

def save_artifact(filename: str, content: str):
    """Save stage output to file."""
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w') as f:
        f.write(content)
    print(f"💾 Saved → {path}")

def human_checkpoint(stage_name: str, output: str, context_key: str) -> str:
    """
    Human-in-the-loop checkpoint.
    Shows output, lets user approve or edit before proceeding.
    Returns the approved (possibly edited) content.
    """
    print("\n" + "="*60)
    print(f"✋ HUMAN CHECKPOINT — {stage_name}")
    print("="*60)
    print("Options:")
    print("  [A] Approve and proceed")
    print("  [E] Edit before proceeding")
    print("  [R] Regenerate with feedback")
    print("="*60)
    
    decision = input("Your choice (A/E/R): ").strip().upper()
    
    if decision == "A":
        context[context_key] = output
        print(f"✅ Approved. Proceeding to next stage.")
        return output
    
    elif decision == "E":
        print("Paste your edited version below. Enter 'DONE' on a new line when finished:")
        lines = []
        while True:
            line = input()
            if line.strip() == "DONE":
                break
            lines.append(line)
        edited = "\n".join(lines)
        context[context_key] = edited
        print(f"✅ Edited version saved. Proceeding.")
        return edited
    
    elif decision == "R":
        feedback = input("What should be different? Provide feedback: ").strip()
        context[context_key] = output  # Store original, flag for regen
        context[f"{context_key}_feedback"] = feedback
        print(f"📝 Feedback noted: '{feedback}'")
        print("Re-run the stage cell above with this feedback incorporated.")
        return output
    
    else:
        print("Invalid input — defaulting to Approve.")
        context[context_key] = output
        return output

print("✅ Setup complete. Client initialized.")
print(f"📁 Artifacts will be saved to: {os.path.abspath(OUTPUT_DIR)}")

---
## 📥 Input: Problem Statement
Enter the product problem or opportunity you want the agent to work through.

In [ ]:
# ── Enter your problem statement here ─────────────────────────
PROBLEM_STATEMENT = input(
    "Enter your product problem statement:\n> "
).strip()

context["problem_statement"] = PROBLEM_STATEMENT

print("\n📌 Problem Statement locked in:")
print(f"   {PROBLEM_STATEMENT}")
print("\nReady to begin Stage 1 →")

---
## Stage 1 — 🔍 Discovery & Ideation

In [ ]:
SYSTEM_DISCOVERY = """You are a senior product manager conducting product discovery. 
Your output is structured, sharp, and grounded in real user and market thinking.
Always output clean markdown with clear headers."""

# Check for regeneration feedback
regen_note = ""
if context.get("discovery_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['discovery_feedback']}"

USER_DISCOVERY = f"""Problem Statement: {context['problem_statement']}{regen_note}

Conduct a thorough product discovery for this problem. Produce:

## 1. Market Context
- Market size estimate (TAM/SAM/SOM)
- Key trends driving this problem
- 3–5 competitive solutions and their gaps

## 2. User Personas
Create 2–3 distinct user personas. For each:
- Name, role, context
- Top 3 pain points
- Jobs-to-be-done
- Current workarounds

## 3. Problem Framing
- Refined problem statement ("How might we...")
- Root cause analysis (5 Whys)
- Key assumptions to validate

## 4. Opportunity Statement
One crisp paragraph: what is the opportunity, for whom, and what would success look like?
"""

print("🔍 Running Stage 1: Discovery & Ideation...")
discovery_output = call_claude(SYSTEM_DISCOVERY, USER_DISCOVERY)
display_output("Stage 1 — Discovery & Ideation", discovery_output)
save_artifact("01_discovery.md", discovery_output)

In [ ]:
# ✋ CHECKPOINT 1
discovery_approved = human_checkpoint(
    "Discovery & Ideation", 
    discovery_output, 
    "discovery"
)

---
## Stage 2 — 🗺️ Roadmapping & Prioritization

In [ ]:
SYSTEM_ROADMAP = """You are a senior product manager building a product roadmap.
Use structured prioritization frameworks. Be opinionated — make real tradeoff decisions.
Output clean markdown with tables where appropriate."""

regen_note = ""
if context.get("roadmap_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['roadmap_feedback']}"

USER_ROADMAP = f"""Problem Statement: {context['problem_statement']}

Discovery Output:
{context['discovery']}{regen_note}

Build a prioritized product roadmap. Produce:

## 1. Feature Ideation
Generate 10–15 potential features or capabilities that address the personas and problems identified.

## 2. RICE Prioritization
Score each feature on:
- Reach (1–10)
- Impact (1–10)  
- Confidence (0.5 / 0.8 / 1.0)
- Effort (person-weeks)
- RICE Score = (Reach × Impact × Confidence) / Effort

Present as a markdown table sorted by RICE score.

## 3. MoSCoW Classification
Classify top features into Must Have / Should Have / Could Have / Won't Have (v1)

## 4. Phased Roadmap
- Phase 1 (MVP — 0–3 months): Core must-haves
- Phase 2 (Growth — 3–6 months): Should-haves + feedback-driven iterations  
- Phase 3 (Scale — 6–12 months): Could-haves + expansion

## 5. Key Tradeoffs & Decisions
What are you explicitly NOT building in v1 and why?
"""

print("🗺️ Running Stage 2: Roadmapping & Prioritization...")
roadmap_output = call_claude(SYSTEM_ROADMAP, USER_ROADMAP)
display_output("Stage 2 — Roadmapping & Prioritization", roadmap_output)
save_artifact("02_roadmap.md", roadmap_output)

In [ ]:
# ✋ CHECKPOINT 2
roadmap_approved = human_checkpoint(
    "Roadmapping & Prioritization",
    roadmap_output,
    "roadmap"
)

---
## Stage 3 — 📋 BRD / FRD Generation

In [ ]:
SYSTEM_BRD = """You are a senior product manager writing formal product requirements.
Your BRD and FRD are precise, unambiguous, and engineer-ready.
Use standard PM documentation conventions. Output clean markdown."""

regen_note = ""
if context.get("brd_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['brd_feedback']}"

USER_BRD = f"""Problem Statement: {context['problem_statement']}

Roadmap (Phase 1 MVP features):
{context['roadmap']}{regen_note}

Generate full BRD and FRD for Phase 1 MVP. Produce:

## BUSINESS REQUIREMENTS DOCUMENT (BRD)

### 1. Executive Summary
### 2. Business Objectives & Success Metrics
### 3. Scope (In / Out of Scope)
### 4. Stakeholder Map
### 5. Business Constraints & Assumptions
### 6. Risk Register (top 5 risks with mitigation)

---

## FUNCTIONAL REQUIREMENTS DOCUMENT (FRD)

For each Phase 1 feature, write:

### Feature: [Name]
**User Story:** As a [persona], I want to [action] so that [outcome].
**Acceptance Criteria:**
- Given / When / Then format for each criterion
**Edge Cases & Error States:**
**Dependencies:**
**Non-Functional Requirements:** (performance, security, accessibility)
"""

print("📋 Running Stage 3: BRD / FRD Generation...")
brd_output = call_claude(SYSTEM_BRD, USER_BRD)
display_output("Stage 3 — BRD / FRD", brd_output)
save_artifact("03_brd_frd.md", brd_output)

In [ ]:
# ✋ CHECKPOINT 3
brd_approved = human_checkpoint(
    "BRD / FRD",
    brd_output,
    "brd"
)

---
## Stage 4 — 🎨 UI/UX & Design Brief

In [ ]:
SYSTEM_UXDESIGN = """You are a senior product manager creating a design brief for a UX team.
You think in flows, states, and user journeys — not just screens.
Your brief is clear enough for a designer to start work without a meeting."""

regen_note = ""
if context.get("ux_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['ux_feedback']}"

USER_UX = f"""Functional Requirements:
{context['brd']}{regen_note}

Create a UI/UX design brief. Produce:

## 1. Design Principles
3–5 guiding principles for this product's design language.

## 2. User Flows
For each key feature, describe the end-to-end user flow:
- Entry point
- Step-by-step journey
- Success state
- Error/empty states

## 3. Screen Inventory
List every screen/view needed for Phase 1 with:
- Screen name
- Primary purpose
- Key components/elements
- Navigation in/out

## 4. Wireframe Descriptions
For each critical screen, describe the layout in enough detail for a designer:
- Header/nav elements
- Primary content area
- CTAs and interactive elements
- Mobile vs desktop considerations

## 5. Accessibility & Inclusivity Requirements
WCAG 2.1 AA compliance notes, color contrast, keyboard nav, screen reader support.

## 6. Open Design Questions
What decisions need designer input before mockups begin?
"""

print("🎨 Running Stage 4: UI/UX & Design Brief...")
ux_output = call_claude(SYSTEM_UXDESIGN, USER_UX)
display_output("Stage 4 — UI/UX & Design Brief", ux_output)
save_artifact("04_ux_design_brief.md", ux_output)

In [ ]:
# ✋ CHECKPOINT 4
ux_approved = human_checkpoint(
    "UI/UX & Design Brief",
    ux_output,
    "ux"
)

---
## Stage 5 — 🧪 UAT Planning & Test Cases

In [ ]:
SYSTEM_UAT = """You are a senior product manager building a UAT plan.
You have run UAT for complex banking and digital products.
Your test cases are exhaustive — you think adversarially about what can break.
Output structured markdown with tables for test case matrices."""

regen_note = ""
if context.get("uat_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['uat_feedback']}"

USER_UAT = f"""BRD/FRD:
{context['brd']}

UX Flows:
{context['ux']}{regen_note}

Build a comprehensive UAT plan. Produce:

## 1. UAT Strategy
- Objectives and scope
- Entry and exit criteria
- Roles and responsibilities (PM, QA, business users, engineers)
- Environment requirements

## 2. Test Case Matrix
For each feature, create test cases covering:
| Test ID | Feature | Scenario | Steps | Expected Result | Pass/Fail |

Cover:
- Happy path (primary flow)
- Alternate paths
- Edge cases
- Error/failure states
- Performance scenarios
- Security/access control

Aim for minimum 20 test cases across features.

## 3. Regression Test Checklist
Key areas to re-test after any fix.

## 4. Bug Triage Framework
- Severity definitions (Critical / High / Medium / Low)
- Resolution SLAs per severity
- Go/No-Go criteria for launch

## 5. UAT Sign-Off Template
Formal sign-off checklist for stakeholders before Production.
"""

print("🧪 Running Stage 5: UAT Planning & Test Cases...")
uat_output = call_claude(SYSTEM_UAT, USER_UAT)
display_output("Stage 5 — UAT Planning & Test Cases", uat_output)
save_artifact("05_uat_plan.md", uat_output)

In [ ]:
# ✋ CHECKPOINT 5
uat_approved = human_checkpoint(
    "UAT Planning & Test Cases",
    uat_output,
    "uat"
)

---
## Stage 6 — 🚀 Production Readiness

In [ ]:
SYSTEM_PROD = """You are a senior product manager leading a production launch.
You think in risk, rollback, and operational stability.
Your production readiness checklist leaves nothing to chance."""

regen_note = ""
if context.get("prod_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['prod_feedback']}"

USER_PROD = f"""Product Summary:
{context['problem_statement']}

UAT Results (approved):
{context['uat']}{regen_note}

Build a production readiness plan. Produce:

## 1. Production Readiness Checklist
Group by category:
- Engineering (code freeze, deployment, infrastructure)
- Data & Security (data migration, encryption, access controls)
- Compliance & Legal (regulatory sign-offs, T&Cs, privacy)
- Operations (runbooks, support docs, escalation paths)
- Monitoring & Alerting (dashboards, SLAs, on-call)
- Communication (internal announcement, external comms)

## 2. Go / No-Go Decision Framework
Table of launch criteria with owner and status column.

## 3. Rollback Plan
- Trigger conditions for rollback
- Step-by-step rollback procedure
- Data integrity considerations
- Communication plan during incident

## 4. Launch Day Runbook
Hour-by-hour plan for launch day with owners and checkpoints.

## 5. Post-Launch Monitoring Plan
- Key metrics to watch in first 24h / 7 days / 30 days
- Alert thresholds
- Feedback collection mechanisms
"""

print("🚀 Running Stage 6: Production Readiness...")
prod_output = call_claude(SYSTEM_PROD, USER_PROD)
display_output("Stage 6 — Production Readiness", prod_output)
save_artifact("06_production_readiness.md", prod_output)

In [ ]:
# ✋ CHECKPOINT 6
prod_approved = human_checkpoint(
    "Production Readiness",
    prod_output,
    "prod"
)

---
## Stage 7 — 📣 GTM Planning

In [ ]:
SYSTEM_GTM = """You are a senior product manager with strong GTM and marketing strategy experience.
You think in segments, positioning, channels, and metrics.
Your GTM plan is actionable and tied to measurable outcomes."""

regen_note = ""
if context.get("gtm_feedback"):
    regen_note = f"\n\nIMPORTANT — Incorporate this feedback: {context['gtm_feedback']}"

USER_GTM = f"""Problem Statement: {context['problem_statement']}

Discovery & Personas:
{context['discovery']}

Roadmap (Phase 1):
{context['roadmap']}{regen_note}

Build a comprehensive GTM plan. Produce:

## 1. Target Segment & ICP
- Ideal Customer Profile (firmographic + behavioral)
- Segment sizing and prioritization
- Early adopter profile

## 2. Positioning & Messaging
- Positioning statement (For [target], [product] is the [category] that [key benefit] unlike [alternative])
- Core value propositions per persona
- Key messages for each channel
- Objection handling guide

## 3. Channel Strategy
- Primary and secondary channels
- Rationale per channel
- Content / campaign ideas per channel

## 4. Launch Timeline
- Pre-launch (T-4 weeks to T-1 week)
- Launch week activities
- Post-launch (30/60/90 day plan)

## 5. Pricing & Packaging Recommendations
- Pricing model options with pros/cons
- Recommended approach with rationale

## 6. Success Metrics & OKRs
- North Star Metric
- OKRs for first 90 days
- Leading vs lagging indicators

## 7. Internal Comms Plan
What does sales, support, and leadership need to know before launch?
"""

print("📣 Running Stage 7: GTM Planning...")
gtm_output = call_claude(SYSTEM_GTM, USER_GTM)
display_output("Stage 7 — GTM Planning", gtm_output)
save_artifact("07_gtm_plan.md", gtm_output)

In [ ]:
# ✋ CHECKPOINT 7
gtm_approved = human_checkpoint(
    "GTM Planning",
    gtm_output,
    "gtm"
)

---
## 📦 Export — Full Artifact Bundle

In [ ]:
# Generate master summary document
SYSTEM_SUMMARY = """You are a senior product manager writing an executive summary.
Be crisp. One page maximum. Tie everything together."""

USER_SUMMARY = f"""Problem Statement: {context['problem_statement']}

Summarize the full product lifecycle output in an executive brief covering:
1. The problem and opportunity (2–3 sentences)
2. What we're building — Phase 1 MVP (3–5 bullet points)
3. Key product decisions and tradeoffs made
4. Launch readiness status
5. North Star Metric and 90-day OKRs
6. Recommended next steps
"""

print("📦 Generating Executive Summary...")
summary_output = call_claude(SYSTEM_SUMMARY, USER_SUMMARY)
display_output("Executive Summary", summary_output)
save_artifact("00_executive_summary.md", summary_output)

# Bundle index
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
index = f"""# PM Lifecycle Agent — Artifact Bundle
**Generated:** {timestamp}  
**Problem:** {context['problem_statement']}

## Artifacts
| File | Stage |
|------|-------|
| 00_executive_summary.md | Executive Summary |
| 01_discovery.md | Discovery & Ideation |
| 02_roadmap.md | Roadmapping & Prioritization |
| 03_brd_frd.md | BRD / FRD |
| 04_ux_design_brief.md | UI/UX & Design Brief |
| 05_uat_plan.md | UAT Planning & Test Cases |
| 06_production_readiness.md | Production Readiness |
| 07_gtm_plan.md | GTM Planning |
"""
save_artifact("README.md", index)

print("\n" + "="*60)
print("🎉 PM LIFECYCLE AGENT COMPLETE")
print("="*60)
print(f"\n📁 All artifacts saved to: {os.path.abspath(OUTPUT_DIR)}")
print(f"📄 8 documents generated")
print(f"✋ 7 human checkpoints passed")
print("\nArtifacts ready for GitHub upload.")